# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print basic dataset information
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

In [ ]:
# List available record sets and their fields by @id
pp = pprint.PrettyPrinter(indent=2)

if hasattr(metadata, 'record_sets'):
    for rs in metadata.record_sets:
        print(f"RecordSet name: {rs.name}, @id: {rs.id}")
        print("  Fields:")
        for field in getattr(rs, 'fields', []):
            print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
        # Show columns if available
        if hasattr(rs, 'columns'):
            print("  Columns:")
            for col in rs.columns:
                print(f"    - {col.name} (@id: {col.id})")
        print()
else:
    # Try for 'recordSet', sometimes singular
    if hasattr(metadata, 'recordSet'):
        record_sets = metadata.recordSet
        if not isinstance(record_sets, list):
            record_sets = [record_sets]
        for rs in record_sets:
            print(f"RecordSet: {getattr(rs, 'name', '')} (@id: {rs.id})")
    else:
        print('No record sets found in this dataset.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Find all record set @id's for extraction
record_set_ids = []
record_set_name_by_id = {}

if hasattr(metadata, 'record_sets'):
    for rs in metadata.record_sets:
        record_set_ids.append(rs.id)
        record_set_name_by_id[rs.id] = rs.name
elif hasattr(metadata, 'recordSet'):
    record_sets = metadata.recordSet
    if not isinstance(record_sets, list):
        record_sets = [record_sets]
    for rs in record_sets:
        record_set_ids.append(rs.id)
        record_set_name_by_id[rs.id] = getattr(rs, 'name', rs.id)

if not record_set_ids:
    print("No available record set IDs for extraction.")
else:
    print(f"RecordSet @ids: {record_set_ids}")

# Load all available record sets into DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"- Loaded {len(df)} records for RecordSet {record_set_id} ({record_set_name_by_id[record_set_id]})")
        else:
            print(f"- No records found for RecordSet {record_set_id} ({record_set_name_by_id[record_set_id]})")
    except Exception as e:
        print(f"Error loading records for RecordSet {record_set_id}: {e}")
        continue

# Show columns of first loaded DataFrame
if dataframes:
    first_rs_id = next(iter(dataframes.keys()))
    print(f"\nColumns for RecordSet {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()
else:
    print('No dataframes available. Please check if the dataset contains records.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example EDA for the first DataFrame
import numpy as np

if dataframes:
    df = dataframes[first_rs_id].copy()
    print(f"Working with RecordSet: {first_rs_id} - {record_set_name_by_id.get(first_rs_id, '')}")

    # Try to select the first numeric column for demo
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        numeric_field = numeric_cols[0]
        print(f"Using numeric field for EDA: {numeric_field}")

        # Filter: value > mean
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records where {numeric_field} > mean ({threshold:.2f}): {len(filtered_df)} rows")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nSample of normalized values for {numeric_field}:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a likely categorical field
        possible_groups = df.select_dtypes(include=['object', 'category']).columns.tolist()
        if possible_groups:
            group_field = possible_groups[0]
            print(f"\nGrouping filtered data by {group_field} and computing stats:")
            grouped = filtered_df.groupby(group_field)[numeric_field].agg(['mean', 'count']).reset_index()
            display(grouped.head())
        else:
            print("No suitable categorical fields to group by.")
    else:
        print("No numeric fields found in this record set for EDA.")
else:
    print('No Dataframes loaded, cannot perform EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Visualization Example
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_cols:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Pairplot if more numeric fields exist
    if len(numeric_cols) >= 2:
        sns.pairplot(df[numeric_cols[:2]])
        plt.suptitle("Pairwise Plot of First Two Numeric Fields", y=1.02)
        plt.show()
else:
    print("No numeric data available for visualization.")

## 6. Conclusion
In this notebook, we've leveraged the `mlcroissant` library to access, overview, and analyze the FAIR^2 dataset "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells". We explored record sets, examined available fields and columns by their `@id`, and demonstrated basic exploratory analysis and visualization.

- Croissant-powered loading enables referencing all elements using `@id`.
- We showed how to filter, normalize, and group data on record sets.
- Visualizations highlighted feature distributions for a deeper understanding of dataset structure.

Explore further with advanced queries, joining across record sets, and domain-specific data science workflows!